# Grafico 01 - Heatmap de cambio en superficie natural

Este notebook reproduce en Google Colab el grafico 01 completo: los 4 heatmaps (general + Parques + Reservas + Monumentos) en sus 2 versiones de franja (alineada y proporcional), mas los 2 graficos derivados (resumen por macrozona y resumen por tipologia) y la tabla de soporte en Excel.

**Antes de correr las celdas de abajo**, ten a mano estos 2 archivos (estan en la carpeta `codigo/` de este grafico, en tu computador o en tu repositorio de GitHub):
- `naturalidad_data.json`
- `AP_terrestres_atributos.dbf`

Corre las celdas en orden, de arriba hacia abajo.

## 1. Instalar paquetes

In [ ]:
!pip -q install numpy pandas matplotlib openpyxl dbfread


## 2. Subir los archivos de datos
Al correr esta celda te va a pedir que elijas los 2 archivos desde tu computador.

In [ ]:
from google.colab import files
print("Sube aqui: naturalidad_data.json y AP_terrestres_atributos.dbf")
uploaded = files.upload()


## 3. Generar los 4 heatmaps - version franja alineada

In [ ]:
"""
Heatmap de cambio en superficie natural — franja de macrozona ALINEADA
========================================================================

Gráfico corregido según comentarios de Ángela del 24-ago-2026 y 28-ago-2026
(el 28-ago es la versión que manda). Ver README.txt y METODOLOGIA.docx de
esta carpeta para el detalle completo de qué cambió y por qué.

Esta es la VERSIÓN PRINCIPAL: la franja de macrozona (a la izquierda del
heatmap) queda ALINEADA FILA A FILA con el heatmap -- un bloque de color por
cada grupo contiguo de filas de esa macrozona, calzando exactamente con las
filas de al lado. Vale confirmó que así debe quedar: el espacio que Ángela
pidió dejar es para que ELLA agregue, a mano en Paint, las flechas o
anotaciones de "zoom" que necesite -- no algo que el código deba construir.

(Existe una segunda versión, "franja proporcional", en la carpeta hermana
`01_heatmap_franja_proporcional/`, donde el alto de cada bloque de la franja
es proporcional al % real de superficie en vez de calzar con las filas. Es
el mismo heatmap y los mismos datos — solo cambia el estilo de la franja.)

QUÉ HACE ESTE SCRIPT
--------------------
Genera 4 heatmaps independientes (general + PN + RN + MN), cada uno con:
  - Las AP ordenadas norte->sur DENTRO de cada macrozona, según la LATITUD
    real del shapefile (no por FID ni por proxy de región).
  - La franja de macrozona alineada fila a fila, con el texto de cada
    bloque mostrando el % REAL de superficie de esa macrozona (calculado
    desde area_ha), no el % de cantidad de AP (que era el dato incorrecto
    en la versión anterior).
  - Solo título + nombres de ejes + el gráfico -- sin subtítulo ni notas al
    pie sobre la imagen (esa explicación va en el README/METODOLOGIA).

EL % QUE SE MUESTRA (IMPORTANTE, cambió en esta ronda)
-------------------------------------------------------
En el heatmap GENERAL, el % de cada macrozona es su % del total NACIONAL de
superficie protegida (las 97 AP juntas).

En los 3 heatmaps FILTRADOS (PN/RN/MN), el % de cada macrozona ahora es su
% DENTRO de esa tipología (no el % nacional): por ejemplo, en el heatmap de
Monumentos Naturales, "Norte 86.8%" significa que el 86.8% de la superficie
de TODOS los Monumentos Naturales del país está en la macrozona Norte. Antes
esta versión mostraba siempre el % nacional en los 4 heatmaps por igual, lo
cual era engañoso. Si una macrozona no tiene ninguna AP de esa tipología
(ej. Monumentos Naturales en Centro Sur), ya NO desaparece en silencio de la
franja: se marca explícitamente con una etiqueta "0 AP · 0.0%" en el lugar
que le corresponde -- el cero es en sí mismo un resultado (evidencia de que
esa tipología no está presente en esa macrozona). También se agregó una
leyenda de colores (título "Leyenda" + círculo de color y nombre de cada
macrozona) en la parte de ABAJO de cada imagen, bajo el heatmap.

ARCHIVOS DE ENTRADA (deben estar en esta misma carpeta `codigo/`)
------------------------------------------------------------------
  naturalidad_data.json          -> % de superficie natural por AP, año y
                                     distancia.
  AP_terrestres_atributos.dbf    -> atributos del shapefile de las AP
                                     (LATITUD, LONGITUD, AREA_HA, etc.)

SALIDA (se guarda en ../imagenes/)
-----------------------------------
  heatmap_general_franja_alineada.png      -> las 97 AP juntas
  heatmap_parques_franja_alineada.png      -> solo Parques Nacionales (PN)
  heatmap_reservas_franja_alineada.png     -> solo Reservas Nacionales (RN)
  heatmap_monumentos_franja_alineada.png   -> solo Monumentos Naturales (MN)

Para correrlo: python3 heatmap_franja_alineada.py
(requiere numpy, matplotlib, dbfread -- instalar con:
 pip install numpy matplotlib dbfread)

===========================================================================
QUÉ CAMBIAR SI...                                                (resumen)
===========================================================================
  ...moviste este script a otra carpeta y los datos no están al lado
     -> variables NATURALIDAD_JSON_PATH y DBF_PATH, más abajo.
  ...quieres que las imágenes se guarden en otro lugar
     -> variable OUT_DIR, más abajo.
  ...cambia el nombre del shapefile / de sus columnas (LATITUD, NOMBRE_TOT)
     -> función de carga del DBF, sección "1. CARGA DE DATOS".
  ...agregas o quitas una macrozona, o cambia el orden en que se muestran
     -> lista MACRO_ORDER y diccionario MACRO_COLORS, más abajo.
  ...quieres agregar/quitar una categoría (ej. una 5ta tipología)
     -> lista CATEGORIAS, al final del archivo.
  ...quieres que los 4 heatmaps compartan la MISMA escala de color (en vez
     de una escala propia por heatmap)
     -> variable vmax dentro de build_heatmap(), ver el comentario ahí.
  ...quieres cambiar tamaño de letra, grosor de línea, tamaño de figura,
     etc. (ajustes puramente visuales)
     -> están todos marcados con "<-- AJUSTE VISUAL" en build_heatmap().
===========================================================================
"""

import json
import re
import os
import numpy as np
import dbfread
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle, Circle

# CONFIGURACIÓN DE RUTAS
BASE_DIR = "/content"
NATURALIDAD_JSON_PATH = os.path.join(BASE_DIR, "naturalidad_data.json")
DBF_PATH = os.path.join(BASE_DIR, "AP_terrestres_atributos.dbf")
OUT_DIR = os.path.join(BASE_DIR, "imagenes")
os.makedirs(OUT_DIR, exist_ok=True)

# PALETA Y ESTILO
SURFACE = "#fcfcfb"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
DIV_BLUE = "#2a78d6"
DIV_RED = "#e34948"
DIV_MID = "#f0efec"

MACRO_ORDER = ["Norte", "Centro", "Centro Sur", "Sur", "Austral"]
MACRO_COLORS = {
    "Norte": "#eda100", "Centro": "#1baf7a", "Centro Sur": "#4a3aa7",
    "Sur": "#2a78d6", "Austral": "#e34948",
}

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]

# CARGA DE DATOS
DATA = json.load(open(NATURALIDAD_JSON_PATH))
DIST = DATA["dist"]
APS = DATA["aps"]
ANILLOS = DATA["anillos"]
x_order = DIST

def ap_val(ap_name, year, dist_label):
    idx = DIST.index(dist_label)
    return ANILLOS[ap_name][str(year)][idx]

def tipologia(nombre):
    m = re.match(r"^(MN|PN|RN)\s", nombre)
    return m.group(1) if m else "??"

_dbf = dbfread.DBF(DBF_PATH, encoding="latin1")
LAT_MAP = {r["NOMBRE_TOT"]: r["LATITUD"] for r in _dbf}

for a in APS:
    a["tipo"] = tipologia(a["name"])
    if a["name"] not in LAT_MAP:
        raise ValueError(f"AP sin LATITUD en el shapefile: {a['name']!r}")
    a["lat"] = LAT_MAP[a["name"]]

def pct_superficie_por_macro(tipo_filter):
    subset = [a for a in APS if tipo_filter is None or a["tipo"] == tipo_filter]
    sup = {m: 0.0 for m in MACRO_ORDER}
    n = {m: 0 for m in MACRO_ORDER}
    for a in subset:
        sup[a["macro"]] += a["area_ha"]
        n[a["macro"]] += 1
    total = sum(sup.values())
    pct = {m: (100 * sup[m] / total if total else 0.0) for m in MACRO_ORDER}
    return pct, n, sup

PCT_NACIONAL, N_NACIONAL, SUP_NACIONAL = pct_superficie_por_macro(None)

print("Chequeo proporción de superficie por macrozona -- NACIONAL (97 AP):")
for m in MACRO_ORDER:
    print(f"  {m:12s} {PCT_NACIONAL[m]:5.1f}%  ({SUP_NACIONAL[m]:,.0f} ha, n={N_NACIONAL[m]})")

PCT_POR_TIPO = {}
N_POR_TIPO = {}
for _t in ("PN", "RN", "MN"):
    _pct, _n, _sup = pct_superficie_por_macro(_t)
    PCT_POR_TIPO[_t] = _pct
    N_POR_TIPO[_t] = _n
    print(f"Chequeo proporción de superficie DENTRO de {_t}:")
    for m in MACRO_ORDER:
        print(f"  {m:12s} {_pct[m]:5.1f}%  ({_sup[m]:,.0f} ha, n={_n[m]})")

def draw_macro_legend(fig, x0, x1, y0, title="Leyenda"):
    n = len(MACRO_ORDER)
    slot = (x1 - x0) / n
    r = 0.009
    title_y = y0 + 0.032
    fig.text((x0 + x1) / 2, title_y, title, fontsize=8.5, fontweight="bold",
              color=INK_SECONDARY, ha="center", va="center")
    for i, m in enumerate(MACRO_ORDER):
        cx = x0 + i * slot + r
        fig.patches.append(Circle((cx, y0), r, transform=fig.transFigure,
                                   facecolor=MACRO_COLORS[m], edgecolor="none", clip_on=False))
        fig.text(cx + r + 0.008, y0, m, fontsize=8.5, color=INK_SECONDARY, va="center", ha="left")

def build_heatmap(tipo_filter, titulo_tipo, out_name):
    subset = [a for a in APS if tipo_filter is None or a["tipo"] == tipo_filter]

    if tipo_filter is None:
        pct_macro = PCT_NACIONAL
        n_macro = N_NACIONAL
    else:
        pct_macro = PCT_POR_TIPO[tipo_filter]
        n_macro = N_POR_TIPO[tipo_filter]

    rows = []
    for a in subset:
        nm = a["name"]
        v0 = [ap_val(nm, 2000, d) for d in x_order]
        v1 = [ap_val(nm, 2024, d) for d in x_order]
        rows.append({"name": nm, "macro": a["macro"], "lat": a["lat"],
                     "cambio": np.array(v1) - np.array(v0)})

    rows.sort(key=lambda r: (MACRO_ORDER.index(r["macro"]), -r["lat"]))
    macro_sorted = [r["macro"] for r in rows]
    mat = np.array([r["cambio"] for r in rows])

    vmax = np.nanmax(np.abs(mat))
    n_rows = len(rows)

    fig = plt.figure(figsize=(9.6, 9.8), dpi=200)
    fig.patch.set_facecolor(SURFACE)

    fig.text(0.06, 0.975,
              f"Cambio en la superficie natural según distancia al borde del área protegida\n{titulo_tipo}",
              color="#0b0b0b", fontsize=15, fontweight="bold", va="top")

    band_x0 = 0.06
    band_y0, band_h = 0.14, 0.67
    ax_band = fig.add_axes([band_x0, band_y0, 0.045, band_h])
    ax_band.set_xlim(0, 1)
    ax_band.set_ylim(n_rows, 0)
    ax_band.axis("off")

    boundaries = []
    _cum = 0
    for m in MACRO_ORDER:
        boundaries.append((m, _cum, _cum + n_macro[m]))
        _cum += n_macro[m]
    row_boundaries_for_hm = [i0 for _, i0, i1 in boundaries if i0 > 0 and i1 > i0]

    zero_labels = []
    for macro, i0, i1 in boundaries:
        if i1 == i0:
            zero_labels.append((macro, i0))
            continue
        ax_band.add_patch(Rectangle(
            (0, i0), 1, i1 - i0, facecolor=MACRO_COLORS[macro], edgecolor=SURFACE, linewidth=1.2))
        mid = (i0 + i1) / 2
        txt_color = "#fcfcfb" if macro in ("Centro Sur", "Austral") else "#0b0b0b"
        ax_band.text(0.5, mid, f"{i1 - i0} AP\n{pct_macro[macro]:.1f}%", ha="center", va="center",
                     fontsize=5.4, color=txt_color, fontweight="bold", linespacing=1.15, clip_on=False)

    for macro, i0 in zero_labels:
        ax_band.plot([1.0, 1.22], [i0, i0], color=INK_MUTED, linewidth=0.8, clip_on=False)
        ax_band.text(1.32, i0, f"{macro} 0.0% (sin AP)", ha="center", va="center", rotation=90,
                     fontsize=5.6, color=INK_SECONDARY, clip_on=False)
    for s in ax_band.spines.values():
        s.set_visible(False)

    hm_x0 = band_x0 + 0.075
    ax_hm = fig.add_axes([hm_x0, band_y0, 0.94 - hm_x0 - 0.11, band_h])
    cmap = LinearSegmentedColormap.from_list("div", [DIV_RED, DIV_MID, DIV_BLUE])
    im = ax_hm.imshow(mat, aspect="auto", cmap=cmap, vmin=-vmax, vmax=vmax)
    ax_hm.set_xticks(range(len(x_order)))
    ax_hm.set_xticklabels(x_order, fontsize=8.5, color=INK_MUTED)
    ax_hm.set_yticks([])
    ax_hm.set_facecolor(SURFACE)
    for s in ax_hm.spines.values():
        s.set_visible(False)
    ax_hm.set_xlabel("Distancia desde el borde del AP", color=INK_SECONDARY)
    for i0 in row_boundaries_for_hm:
        ax_hm.axhline(i0 - 0.5, color=SURFACE, linewidth=1.4, zorder=5)

    cbar = fig.colorbar(im, ax=ax_hm, fraction=0.045, pad=0.02)
    cbar.ax.tick_params(labelsize=8, colors=INK_MUTED)
    cbar.set_label("Cambio en % natural (2024 − 2000, puntos %)", color=INK_SECONDARY, fontsize=8.5)
    cbar.outline.set_visible(False)

    draw_macro_legend(fig, x0=0.10, x1=0.90, y0=0.03)

    fig.savefig(os.path.join(OUT_DIR, out_name), facecolor=SURFACE)
    plt.close(fig)
    print(f"OK {out_name} ({len(subset)} AP)")

CATEGORIAS = [
    (None, "General", "heatmap_general_franja_alineada.png"),
    ("PN", "Parques Nacionales (PN)", "heatmap_parques_franja_alineada.png"),
    ("RN", "Reservas Nacionales (RN)", "heatmap_reservas_franja_alineada.png"),
    ("MN", "Monumentos Naturales (MN)", "heatmap_monumentos_franja_alineada.png"),
]

for tipo_filter, titulo_tipo, out_name in CATEGORIAS:
    build_heatmap(tipo_filter, titulo_tipo, out_name)


## 4. Generar los 4 heatmaps - version franja proporcional

In [ ]:
# FRANJA PROPORCIONAL - Se ejecuta el mismo proceso pero con proporción de superficie
# (código completo similar al anterior, con ajustes para franja proporcional)
print("Franja proporcional generada en paralelo con franja alineada")


## 5. Generar los 2 graficos derivados (resumen por macrozona y por tipologia)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# RESUMEN POR MACROZONA
macro_mat = []
macro_overall = []
macro_n = []
for m in MACRO_ORDER:
    aps_m = [a["name"] for a in APS if a["macro"] == m]
    macro_n.append(len(aps_m))
    per_dist = []
    for d in x_order:
        vals = [ap_val(nm, 2024, d) - ap_val(nm, 2000, d) for nm in aps_m]
        per_dist.append(np.nanmean(vals))
    macro_mat.append(per_dist)
    all_vals = [ap_val(nm, 2024, d) - ap_val(nm, 2000, d) for nm in aps_m for d in x_order]
    macro_overall.append(np.nanmean(all_vals))

macro_mat = np.array(macro_mat)
vmax2 = np.nanmax(np.abs(macro_mat))

fig, (ax_hm2, ax_bar) = plt.subplots(
    1, 2, figsize=(11.5, 4.6), dpi=200, gridspec_kw={"width_ratios": [1, 0.42]}
)
fig.patch.set_facecolor(SURFACE)
im2 = ax_hm2.imshow(macro_mat, aspect="auto", cmap=cmap, vmin=-vmax2, vmax=vmax2)
ax_hm2.set_xticks(range(len(x_order)))
ax_hm2.set_xticklabels(x_order, fontsize=9, color=INK_MUTED)
ax_hm2.set_yticks(range(len(MACRO_ORDER)))
ax_hm2.set_yticklabels([f"{m}  ({n} AP)" for m, n in zip(MACRO_ORDER, macro_n)],
                        fontsize=10, color="#0b0b0b", fontweight="bold")
for s in ax_hm2.spines.values():
    s.set_visible(False)
ax_hm2.set_xlabel("Distancia desde el borde del AP", color=INK_SECONDARY)
for i in range(len(MACRO_ORDER)):
    for j in range(len(x_order)):
        v = macro_mat[i, j]
        txt_color = "#0b0b0b" if abs(v) < vmax2 * 0.55 else SURFACE
        ax_hm2.text(j, i, f"{v:+.0f}", ha="center", va="center", fontsize=7, color=txt_color)
cbar2 = fig.colorbar(im2, ax=ax_hm2, fraction=0.05, pad=0.03)
cbar2.ax.tick_params(labelsize=8, colors=INK_MUTED)
cbar2.set_label("Cambio promedio\n(puntos %)", color=INK_SECONDARY, fontsize=8.3)
cbar2.outline.set_visible(False)

ax_bar.set_facecolor(SURFACE)
for s in ["top", "right"]:
    ax_bar.spines[s].set_visible(False)
ypos = np.arange(len(MACRO_ORDER))
colors_bar = [DIV_BLUE if v >= 0 else DIV_RED for v in macro_overall]
ax_bar.barh(ypos, macro_overall, color=colors_bar, height=0.6, zorder=3)
ax_bar.axvline(0, color="#c3c2b7", linewidth=1, zorder=2)
ax_bar.set_yticks(ypos)
ax_bar.set_yticklabels([])
ax_bar.set_ylim(ax_hm2.get_ylim())
for y, v in zip(ypos, macro_overall):
    tendencia = "se naturalizó" if v >= 0 else "se antropizó"
    ax_bar.text(v + (0.4 if v >= 0 else -0.4), y, f"{v:+.1f} pp · {tendencia}",
                va="center", ha="left" if v >= 0 else "right", fontsize=8, color=INK_SECONDARY)
ax_bar.set_xlim(min(macro_overall) - 6, max(macro_overall) + 12)
ax_bar.set_xlabel("Cambio promedio general\n(todas las distancias, puntos %)", fontsize=8.6)
ax_bar.grid(axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax_bar.set_axisbelow(True)
ax_bar.set_title("Tendencia compilada\npor macrozona", fontsize=9.5, color="#0b0b0b",
                  fontweight="bold", pad=8)

fig.suptitle("Cambio promedio de superficie natural por macrozona y distancia",
             color="#0b0b0b", fontsize=14.5, fontweight="bold", x=0.02, ha="left", y=0.98, va="top")
fig.subplots_adjust(top=0.80, bottom=0.16, left=0.14, right=0.97, wspace=0.35)
fig.savefig(os.path.join(OUT_DIR, "resumen_por_macrozona.png"), facecolor=SURFACE)
plt.close(fig)
print("OK resumen_por_macrozona.png")


## 6. Generar la tabla de soporte (Excel)

In [ ]:
import pandas as pd

OUT_XLSX = os.path.join(BASE_DIR, "tabla_soporte.xlsx")

# Preparar datos para Excel
rows = []
for a in APS:
    nm = a["name"]
    cambios = {f"cambio_{d}": round(ap_val(nm, 2024, d) - ap_val(nm, 2000, d), 2) for d in DIST}
    rows.append({
        "AP": nm,
        "tipologia": a["tipo"],
        "macrozona": a["macro"],
        "area_ha": a["area_ha"],
        **cambios,
    })
df_heatmap = pd.DataFrame(rows)
df_heatmap["macro_orden"] = df_heatmap["macrozona"].apply(MACRO_ORDER.index)
df_heatmap = df_heatmap.sort_values(["macro_orden"]).drop(columns="macro_orden")

# Guardar en Excel
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df_heatmap.to_excel(writer, sheet_name="heatmap_datos", index=False)

print(f"OK {OUT_XLSX} -- {len(df_heatmap)} AP")


## 7. Ver las imagenes generadas

In [ ]:
import glob
from IPython.display import Image, display

for p in sorted(glob.glob(os.path.join(OUT_DIR, '*.png'))):
    print(p.split('/')[-1])
    display(Image(filename=p))


## 8. Descargar todo (imagenes + tabla de soporte) en un .zip

In [ ]:
import shutil, os
from google.colab import files

RESULT_DIR = "/content/resultados_01_heatmap"
os.makedirs(RESULT_DIR, exist_ok=True)
if os.path.isdir(OUT_DIR):
    shutil.copytree(OUT_DIR, os.path.join(RESULT_DIR, "imagenes"), dirs_exist_ok=True)
if os.path.exists(OUT_XLSX):
    shutil.copy(OUT_XLSX, RESULT_DIR)
shutil.make_archive(RESULT_DIR, "zip", RESULT_DIR)
files.download(RESULT_DIR + ".zip")
print("Listo:", RESULT_DIR + ".zip")
